# Week 4 Practical: Graph Neural Networks for Molecular Property Prediction
**AI for Drug Discovery**

In this practical, you will:
1. Represent molecules as graphs
2. Train a Graph Convolutional Network (GCN) using DeepChem
3. Compare GNN performance to your Random Forest baseline
4. Analyze when GNNs outperform classical ML

In [ ]:
# Install dependencies (this may take a few minutes)
!pip install deepchem rdkit-pypi scikit-learn pandas matplotlib seaborn -q

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import deepchem as dc
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

sns.set_style('whitegrid')
print(f'DeepChem version: {dc.__version__}')
print('All imports successful!')

## 1. Load a MoleculeNet Dataset
We'll use the **BACE** dataset: 1,513 compounds tested against BACE-1, a key target for Alzheimer's disease.

**Reference:** Wu, Z. et al. (2018). MoleculeNet: A Benchmark for Molecular Machine Learning. Chemical Science 9:513-530

In [ ]:
# Load BACE dataset with scaffold split
tasks, datasets, transformers = dc.molnet.load_bace_classification(
    featurizer='GraphConv',
    splitter='scaffold'
)

train_dataset, valid_dataset, test_dataset = datasets

print(f'Task: {tasks}')
print(f'Training set: {len(train_dataset)} molecules')
print(f'Validation set: {len(valid_dataset)} molecules')
print(f'Test set: {len(test_dataset)} molecules')

## 2. Visualize Some Molecules from the Dataset

In [ ]:
# Load SMILES for visualization
try:
    tasks2, datasets2, _ = dc.molnet.load_bace_classification(
        featurizer='ECFP', splitter='scaffold')
    smiles_list = datasets2[0].ids[:12]
    mols = [Chem.MolFromSmiles(s) for s in smiles_list if Chem.MolFromSmiles(s) is not None]
    img = Draw.MolsToGridImage(mols[:12], molsPerRow=4, subImgSize=(300, 250))
    display(img)
except Exception as e:
    print(f'Visualization skipped: {e}')

## 3. Train a Graph Convolutional Network (GCN)
DeepChem's GraphConvModel implements a simple GCN architecture.

**Reference:** Kipf, T.N. & Welling, M. (2017). Semi-Supervised Classification with Graph Convolutional Networks. ICLR

In [ ]:
# Build and train GCN model
model_gcn = dc.models.GraphConvModel(
    n_tasks=1,
    mode='classification',
    dropout=0.2,
    batch_size=64,
    learning_rate=0.001
)

# Train for 50 epochs
print('Training GCN...')
losses = []
for epoch in range(50):
    loss = model_gcn.fit(train_dataset, nb_epoch=1)
    losses.append(loss)
    if (epoch + 1) % 10 == 0:
        train_score = model_gcn.evaluate(train_dataset, [dc.metrics.Metric(dc.metrics.roc_auc_score)])
        valid_score = model_gcn.evaluate(valid_dataset, [dc.metrics.Metric(dc.metrics.roc_auc_score)])
        print(f'Epoch {epoch+1}: Train AUC={list(train_score.values())[0]:.3f}, '
              f'Valid AUC={list(valid_score.values())[0]:.3f}')

print('Training complete!')

In [ ]:
# Evaluate GCN on test set
test_score_gcn = model_gcn.evaluate(test_dataset, [dc.metrics.Metric(dc.metrics.roc_auc_score)])
print(f'GCN Test AUC-ROC: {list(test_score_gcn.values())[0]:.3f}')

## 4. Train a Random Forest Baseline
Using Morgan fingerprints for fair comparison.

In [ ]:
# Load ECFP fingerprints for RF
tasks_ecfp, datasets_ecfp, transformers_ecfp = dc.molnet.load_bace_classification(
    featurizer='ECFP',
    splitter='scaffold'
)
train_ecfp, valid_ecfp, test_ecfp = datasets_ecfp

# Train Random Forest
rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
rf.fit(train_ecfp.X, train_ecfp.y.ravel())

# Evaluate
y_pred_rf = rf.predict_proba(test_ecfp.X)[:, 1]
auc_rf = roc_auc_score(test_ecfp.y.ravel(), y_pred_rf)
print(f'Random Forest Test AUC-ROC: {auc_rf:.3f}')

## 5. Compare Models

In [ ]:
# Summary comparison
results = pd.DataFrame({
    'Model': ['Random Forest (ECFP4)', 'Graph Conv Network'],
    'Test AUC-ROC': [auc_rf, list(test_score_gcn.values())[0]]
})
print(results.to_string(index=False))
print()
if list(test_score_gcn.values())[0] > auc_rf:
    print('GCN outperforms RF on this dataset!')
else:
    print('RF is competitive with (or better than) GCN on this dataset.')
    print('This is common for smaller datasets like BACE (~1500 molecules).')

In [ ]:
# Plot training loss curve
plt.figure(figsize=(10, 5))
plt.plot(losses, color='#1f77b4', linewidth=1)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Training Loss', fontsize=12)
plt.title('GCN Training Loss', fontsize=14)
plt.tight_layout()
plt.show()

## 6. (Bonus) Try the HIV Dataset
The HIV dataset has ~41,000 molecules — GNNs should have a bigger advantage on larger datasets.

In [ ]:
# Uncomment to try the HIV dataset (takes longer to train)
# tasks_hiv, datasets_hiv, _ = dc.molnet.load_hiv(
#     featurizer='GraphConv', splitter='scaffold')
# train_hiv, valid_hiv, test_hiv = datasets_hiv
# print(f'HIV dataset: {len(train_hiv)} train, {len(test_hiv)} test')
#
# model_hiv = dc.models.GraphConvModel(n_tasks=1, mode='classification',
#                                       dropout=0.2, learning_rate=0.001)
# model_hiv.fit(train_hiv, nb_epoch=30)
# score_hiv = model_hiv.evaluate(test_hiv, [dc.metrics.Metric(dc.metrics.roc_auc_score)])
# print(f'HIV GCN Test AUC: {list(score_hiv.values())[0]:.3f}')

## 7. Exercises

1. **Hyperparameter tuning**: Try different learning rates (0.0001, 0.001, 0.01), dropout rates, and number of epochs
2. **Deeper model**: Add more graph conv layers. Does it help?
3. **Different dataset**: Try the Tox21 dataset (`dc.molnet.load_tox21`)
4. **Chemprop**: Install and try Chemprop (`pip install chemprop`) — how does it compare?

## References
- Wu, Z. et al. (2018). MoleculeNet: A Benchmark for Molecular Machine Learning. Chemical Science 9:513-530
- Kipf, T.N. & Welling, M. (2017). Semi-Supervised Classification with Graph Convolutional Networks. ICLR
- Yang, K. et al. (2019). Analyzing Learned Molecular Representations for Property Prediction. JCIM 59:3370-3388
- Stokes, J.M. et al. (2020). A Deep Learning Approach to Antibiotic Discovery. Cell 180:688-702